# Near-field propagation conventions

Executable companion to the near-field contract. These local reference functions will be replaced by the public propagators in issue #74. The notebook checks centred FFT conventions, the Fresnel sampling bound, flux conservation, paraxial agreement, and Gaussian-beam spreading.

In [ ]:
import torch
import matplotlib.pyplot as plt
from fiatlux.core import Grid

torch.set_default_dtype(torch.float64)

wavelength = 632.8e-9
distance = 0.05
waist_radius = 0.15e-3

grid = Grid(nx=128, ny=128, dx=25e-6, dy=25e-6, dtype=torch.float64)
x, y = grid.meshgrid()
radius_squared = x.square() + y.square()
input_field = torch.exp(-radius_squared / waist_radius**2).to(torch.complex128)
input_field /= (
    input_field.abs().square().sum() * grid.dx * grid.dy
).sqrt()

## Sampling

Same-grid Fresnel transfer-function propagation requires

\[
|z| \leq \min(n_x dx^2/\lambda, n_y dy^2/\lambda).
\]

Angular-spectrum propagation retains only spatial frequencies satisfying \(f_x^2+f_y^2\leq1/\lambda^2\) when evanescent waves are discarded.

In [ ]:
limit = min(
    grid.nx * grid.dx**2 / wavelength,
    grid.ny * grid.dy**2 / wavelength,
)
fx = torch.fft.fftshift(torch.fft.fftfreq(grid.nx, d=grid.dx))
fy = torch.fft.fftshift(torch.fft.fftfreq(grid.ny, d=grid.dy))
fy_grid, fx_grid = torch.meshgrid(fy, fx, indexing="ij")
frequency_squared = fx_grid.square() + fy_grid.square()
propagating = frequency_squared <= wavelength**-2

print(f"distance = {distance:.3f} m; Fresnel limit = {limit:.3f} m")
assert abs(distance) <= limit

## Reference propagation

The field is centred in Fiatlux, while the FFT origin is stored at index zero. The shifts below are therefore part of the contract. Unitary FFT normalization preserves integrated flux.

In [ ]:
def fft2_centered(field):
    return torch.fft.fftshift(
        torch.fft.fft2(torch.fft.ifftshift(field), norm="ortho")
    )


def ifft2_centered(spectrum):
    return torch.fft.fftshift(
        torch.fft.ifft2(torch.fft.ifftshift(spectrum), norm="ortho")
    )


def angular_spectrum(field, wavelength, distance):
    fz = torch.sqrt((wavelength**-2 - frequency_squared).clamp_min(0))
    transfer = torch.exp(2j * torch.pi * distance * fz)
    transfer = torch.where(propagating, transfer, 0.0)
    return ifft2_centered(fft2_centered(field) * transfer)


def fresnel(field, wavelength, distance):
    carrier = torch.exp(
        torch.as_tensor(2j * torch.pi * distance / wavelength, dtype=field.dtype)
    )
    transfer = carrier * torch.exp(
        -1j * torch.pi * wavelength * distance * frequency_squared
    )
    return ifft2_centered(fft2_centered(field) * transfer)


angular_field = angular_spectrum(input_field, wavelength, distance)
fresnel_field = fresnel(input_field, wavelength, distance)

## Physical checks

For this paraxial Gaussian, both transfer functions should conserve flux, agree in intensity, and reproduce the analytical Gaussian-beam radius.

In [ ]:
def flux(field):
    return field.abs().square().sum() * grid.dx * grid.dy


angular_flux = flux(angular_field)
fresnel_flux = flux(fresnel_field)
torch.testing.assert_close(angular_flux, torch.tensor(1.0), rtol=1e-10, atol=1e-12)
torch.testing.assert_close(fresnel_flux, torch.tensor(1.0), rtol=1e-10, atol=1e-12)

angular_intensity = angular_field.abs().square()
fresnel_intensity = fresnel_field.abs().square()
relative_difference = (
    torch.linalg.vector_norm(angular_intensity - fresnel_intensity)
    / torch.linalg.vector_norm(angular_intensity)
)
assert relative_difference < 5e-3

rayleigh_range = torch.pi * waist_radius**2 / wavelength
expected_radius = waist_radius * torch.sqrt(
    torch.tensor(1 + (distance / rayleigh_range) ** 2)
)
measured_radius = torch.sqrt(
    2 * (radius_squared * fresnel_intensity).sum() / fresnel_intensity.sum()
)
torch.testing.assert_close(measured_radius, expected_radius, rtol=2e-2, atol=0)

print(f"angular-spectrum flux = {float(angular_flux):.12f}")
print(f"Fresnel flux = {float(fresnel_flux):.12f}")
print(f"relative intensity difference = {float(relative_difference):.3e}")
print(
    f"Gaussian radius: measured {float(measured_radius * 1e3):.4f} mm; "
    f"expected {float(expected_radius * 1e3):.4f} mm"
)

## Visual comparison

All three images use the same transverse coordinates: the initial implementations do not resample implicitly.

In [ ]:
extent_mm = [
    float(grid.x.min() * 1e3),
    float(grid.x.max() * 1e3),
    float(grid.y.min() * 1e3),
    float(grid.y.max() * 1e3),
]
images = [input_field.abs().square(), angular_intensity, fresnel_intensity]
titles = ["Input", "Angular spectrum", "Fresnel"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, image, title in zip(axes, images, titles):
    displayed = ax.imshow(
        image.detach().cpu().numpy(), origin="lower", extent=extent_mm
    )
    ax.set_title(title)
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")
    fig.colorbar(displayed, ax=ax)
plt.tight_layout()
plt.show()

center = grid.ny // 2
plt.figure(figsize=(7, 4))
plt.plot(grid.x.numpy() * 1e3, angular_intensity[center].numpy(), label="Angular")
plt.plot(grid.x.numpy() * 1e3, fresnel_intensity[center].numpy(), "--", label="Fresnel")
plt.xlabel("x [mm]")
plt.ylabel("Intensity")
plt.grid(True)
plt.legend()
plt.show()

The production implementation must preserve these results while adding wavelength batching, transfer-function caching, explicit evanescent and anti-aliasing options, actionable sampling errors, autograd, and rectangular-grid validation.